In [1]:
import pandas as pd

# Carico i 2 dataset
festival = pd.read_excel('~/EPICODE/Progetto Finale Looker/drive-download-20250804T194039Z-1-001/dati-festival-sanremo-1951-2023.xlsx')

classifica = pd.read_excel('~/EPICODE/Progetto Finale Looker/drive-download-20250804T194039Z-1-001/Ufficiale/dati-classifica-sanremo-1951-2023.xlsx')

display(festival.head(), classifica.head())

,Anno,Periodo,Sede,Presentatore,Partecipanti,Vincitore
0,1951,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,3,Nilla Pizzi
1,1952,28-30 gennaio,Casinò di Sanremo,Nunzio Filogamo,5,Nilla Pizzi
2,1953,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,10,Carla Boni
3,1954,28-30 gennaio,Casinò di Sanremo,Nunzio Filogamo,12,Gino Latilla
4,1955,27-29 gennaio,Casinò di Sanremo,Armando Pizzo,15,Claudio Villa


,Unnamed: 0,Posizione,Interprete,Canzone,anno,Autori
0,0,1º,Nilla Pizzi,Grazie dei fiori,1951,Gian Carlo Testoni e Mario Panzeri
1,1,2º,Nilla Pizzi e Achille Togliani,La luna si veste d'argento,1951,Biri
2,2,3º,Achille Togliani,Serenata a nessuno,1951,Walter Colì
3,3,F,Achille Togliani e Duo Fasano,Al mercato di Pizzighettone,1951,Aldo Locatelli
4,4,F,Nilla Pizzi e Achille Togliani,Eco tra gli abeti,1951,Enzo Bonagura


In [39]:
print(f'FESTIVAL: {festival.shape}')
print(f'CLASSIFICA: {classifica.shape}')

print('\nCOLONNE FESTIVAL:\n', list(festival.columns))
print('\nCOLONNE CLASSIFICA:\n', list(classifica.columns))

print(F'\nFESTIVAL INFO: \n{festival.dtypes}\nCLASSIFICA INFO: \n{classifica.dtypes}')

FESTIVAL: (73, 6)
CLASSIFICA: (1663, 6)

COLONNE FESTIVAL:
 ['anno', 'periodo', 'sede', 'presentatore', 'partecipanti', 'vincitore']

COLONNE CLASSIFICA:
 ['posizione', 'interprete', 'canzone', 'anno', 'autori', 'posizione_num']

FESTIVAL INFO: 
anno            datetime64[ns]
periodo                 object
sede                    object
presentatore            object
partecipanti             int64
vincitore               object
dtype: object
CLASSIFICA INFO: 
posizione                object
interprete               object
canzone                  object
anno             datetime64[ns]
autori                   object
posizione_num           float64
dtype: object


In [3]:
#Modifico in minuscolo i nomi di tutte le colonne per comodità
festival.columns = festival.columns.str.strip().str.lower()
classifica.columns = classifica.columns.str.strip().str.lower()

***

### ANALISI E MODIFICHE TABELLA 'CLASSIFICA'

In [4]:
#controllo se ci sono duplicati
print('Duplicati in classifica:', classifica.duplicated().sum())

#controllo se ci sono nulli
print('\n','Classifica: \n', classifica.isnull().sum())

Duplicati in classifica: 0

 Classifica: 
 unnamed: 0     0
posizione     10
interprete     0
canzone        0
anno           0
autori         0
dtype: int64


In [5]:
#Controllo i valori nulli all'interno della colonna "Posizione"
classifica.loc[classifica["posizione"].isna()]

,unnamed: 0,posizione,interprete,canzone,anno,autori
1309,1310,NaN,Amalia Gré,Amami per sempre,2007,"A. Grezio, M. Ranauro e P. Palma"
1310,1311,NaN,Marcella e Gianni Bella,Forever per sempre,2007,Mogol e G. Bella
1311,1312,NaN,Stadio,Guardami,2007,S. Grandi e G. Curreri
1312,1313,NaN,Paolo Rossi,In Italia si sta male (si sta bene anziché no),2007,R. Gaetano
1313,1314,NaN,Nada,Luna in piena,2007,N. Malanima
1314,1315,NaN,Johnny Dorelli,Meglio così,2007,G. Calabrese e G. Ferrio
1315,1316,NaN,Fabio Concato,Oltre il giardino,2007,F. Concato
1316,1317,NaN,Leda Battisti,Senza me ti pentirai,2007,L. Battisti e A. Battaglia
1317,1318,NaN,Milva,The Show Must Go On,2007,G. Faletti
1318,1319,NaN,Velvet,Tutto da rifare,2007,"A. Sgreccia, G. Cornetta, P. Ferrantini e P. B..."


In [6]:
#SOSTITUISCO TUTTI I VALORI NULL 

#Trovo i NaN nella colonna Posizione
mask = classifica["posizione"].isna()

#Conto quanti NaN ci sono
n_nan = mask.sum()

#Creo la sequenza che parte da 11 e lunga quanto i NaN
nuovi_valori = range(11, 11 + n_nan)

#Sostituisco i NaN con i nuovi valori
classifica.loc[mask, "posizione"] = nuovi_valori

In [7]:
#Elimino la colonna Unnamed: 0
classifica.drop(columns=['unnamed: 0'], inplace=True)

In [8]:
#Cambio il formato della colonna anno da int a datetime
classifica['anno'] = pd.to_datetime(classifica['anno'].astype(str), format='%Y')
festival['anno'] = pd.to_datetime(festival['anno'].astype(str), format='%Y')

In [9]:
#Estraggo numero da 'Posizione' (es. "1º" -> 1, "F"/"NF" -> NaN)
classifica["posizione_num"] = (
    classifica["posizione"].astype(str)
                       .str.extract(r"(\d+)", expand=False)
                       .astype(float)
)

print(classifica['posizione_num'].dtype)  
classifica.head(5)

float64


,posizione,interprete,canzone,anno,autori,posizione_num
0,1º,Nilla Pizzi,Grazie dei fiori,1951-01-01,Gian Carlo Testoni e Mario Panzeri,1.0
1,2º,Nilla Pizzi e Achille Togliani,La luna si veste d'argento,1951-01-01,Biri,2.0
2,3º,Achille Togliani,Serenata a nessuno,1951-01-01,Walter Colì,3.0
3,F,Achille Togliani e Duo Fasano,Al mercato di Pizzighettone,1951-01-01,Aldo Locatelli,NaN
4,F,Nilla Pizzi e Achille Togliani,Eco tra gli abeti,1951-01-01,Enzo Bonagura,NaN


In [22]:
#Dizionario connettori -> virgola
sostituzioni= {
    r'(?i)\be\b'            : ',',
    r'(?i)\bed\b'           : ',',
    r'(?i)\bcon\b'          : ',',
    r'(?i)\bfeat\.?\b'      : ',',
    r'(?i)\bfeaturing\b'    : ',',
    r'(?i)\band\b'          : ',',
    r'(?i)\bwith\b'         : ',',
    r'&'                    : ',',
    r' / '                  : ',',    # con spazi
    r'/'                    : ',',    # senza spazi
    r'\s[-–]\s'             : ',',    # trattino SOLO se ha spazi ai lati
}

#1) Normalizzo la colonna 'interprete'
classifica['interprete'] = classifica['interprete'].replace(sostituzioni, regex=True).str.strip()

#2) Explode (una riga per interprete)
interpreti_exploded = (
    classifica
      .assign(interprete_expl = lambda d: d['interprete'].str.split(','))  # liste
      .explode('interprete_expl', ignore_index=True)                           # 1 nome/rig
      .assign(interprete_norm = lambda d: d['interprete_expl'].str.strip())    # trim
      .loc[lambda d: d['interprete_expl'].notna() & (d['interprete_expl'] != "")]  # no vuoti
      [ ['anno','canzone','interprete','interprete_expl','posizione','posizione_num'] ]
)

print("Interpreti esplosi:", len(interpreti_exploded))
interpreti_exploded.head(10)

Interpreti esplosi: 2280


,anno,canzone,interprete,interprete_expl,posizione,posizione_num
0,1951-01-01,Grazie dei fiori,Nilla Pizzi,Nilla Pizzi,1º,1.0
1,1951-01-01,La luna si veste d'argento,"Nilla Pizzi , Achille Togliani",Nilla Pizzi,2º,2.0
2,1951-01-01,La luna si veste d'argento,"Nilla Pizzi , Achille Togliani",Achille Togliani,2º,2.0
3,1951-01-01,Serenata a nessuno,Achille Togliani,Achille Togliani,3º,3.0
4,1951-01-01,Al mercato di Pizzighettone,"Achille Togliani , Duo Fasano",Achille Togliani,F,NaN
5,1951-01-01,Al mercato di Pizzighettone,"Achille Togliani , Duo Fasano",Duo Fasano,F,NaN
6,1951-01-01,Eco tra gli abeti,"Nilla Pizzi , Achille Togliani",Nilla Pizzi,F,NaN
7,1951-01-01,Eco tra gli abeti,"Nilla Pizzi , Achille Togliani",Achille Togliani,F,NaN
8,1951-01-01,Famme durmì,"Achille Togliani , Duo Fasano",Achille Togliani,F,NaN
9,1951-01-01,Famme durmì,"Achille Togliani , Duo Fasano",Duo Fasano,F,NaN


In [30]:
#Replico lo stesso per 'autore'
classifica['autori'] = classifica['autori'].replace(sostituzioni, regex=True).str.strip()

autori_exploded = (
    classifica
      .assign(autori_expl = lambda d: d['autori'].str.split(','))
      .explode('autori_expl', ignore_index=True)
      .assign(autore_norm = lambda d: d['autori_expl'].str.strip())
      .loc[lambda d: d['autori_expl'].notna() & (d['autori_expl'] != "")]
      .drop_duplicates(subset=['anno','canzone','autori_expl'])
      [ ['anno','canzone','autori','autori_expl','posizione','posizione_num'] ]
)

print('Autori esplosi:', len(autori_exploded))
autori_exploded.head(10)

Autori esplosi: 3966


,anno,canzone,autori,autori_expl,posizione,posizione_num
0,1951-01-01,Grazie dei fiori,"Gian Carlo Testoni , Mario Panzeri",Gian Carlo Testoni,1º,1.0
1,1951-01-01,Grazie dei fiori,"Gian Carlo Testoni , Mario Panzeri",Mario Panzeri,1º,1.0
2,1951-01-01,La luna si veste d'argento,Biri,Biri,2º,2.0
3,1951-01-01,Serenata a nessuno,Walter Colì,Walter Colì,3º,3.0
4,1951-01-01,Al mercato di Pizzighettone,Aldo Locatelli,Aldo Locatelli,F,NaN
5,1951-01-01,Eco tra gli abeti,Enzo Bonagura,Enzo Bonagura,F,NaN
6,1951-01-01,Famme durmì,Danpa,Danpa,F,NaN
7,1951-01-01,La cicogna distratta,"Aldo Valleroni , Da Rovere",Aldo Valleroni,F,NaN
8,1951-01-01,La cicogna distratta,"Aldo Valleroni , Da Rovere",Da Rovere,F,NaN
9,1951-01-01,La margherita,Ester B. Valdes,Ester B. Valdes,F,NaN


***

### ANALISI E MODIFICHE TABELLA 'FESTIVAL'

In [33]:
#controllo se ci sono duplicati
print('Duplicati in festival:', festival.duplicated().sum())

#controllo se ci sono nulli
print('\n','Festival: \n', festival.isnull().sum())

Duplicati in festival: 0

 Festival: 
 anno            0
periodo         0
sede            0
presentatore    0
partecipanti    0
vincitore       0
dtype: int64


In [36]:
#Elimino tutti i valori nelle parentesi nella colonna 'presentatore'
festival['presentatore'] = festival['presentatore'].str.split('(', n=1).str[0].str.strip()

festival.head(15)

,anno,periodo,sede,presentatore,partecipanti,vincitore
0,1951-01-01,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,3,Nilla Pizzi
1,1952-01-01,28-30 gennaio,Casinò di Sanremo,Nunzio Filogamo,5,Nilla Pizzi
2,1953-01-01,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,10,Carla Boni
3,1954-01-01,28-30 gennaio,Casinò di Sanremo,Nunzio Filogamo,12,Gino Latilla
4,1955-01-01,27-29 gennaio,Casinò di Sanremo,Armando Pizzo,15,Claudio Villa
5,1956-01-01,8-10 marzo,Casinò di Sanremo,Fausto Tommei,6,Franca Raimondi
6,1957-01-01,7-9 febbraio,Casinò di Sanremo,Nunzio Filogamo,17,Claudio Villa
7,1958-01-01,30 gennaio - 1º febbraio,Casinò di Sanremo,Gianni Agus,15,Domenico Modugno
8,1959-01-01,29-31 gennaio,Casinò di Sanremo,Enzo Tortora,17,Domenico Modugno
9,1960-01-01,28-30 gennaio,Casinò di Sanremo,Paolo Ferrari,23,Tony Dallara


***

## Scarico tutti i CSV

In [40]:
classifica.to_csv("sanremo_classifica_clean.csv", index=False, encoding="utf-8-sig")
interpreti_exploded.to_csv("sanremo_interpreti_exploded.csv", index=False, encoding="utf-8-sig")
autori_exploded.to_csv("sanremo_autori_exploded.csv", index=False, encoding="utf-8-sig")
festival.to_csv("sanremo_festival_clean.csv", index=False, encoding="utf-8-sig")